# Assignment 2 — Data Understanding (EDA)

VU Data Mining Techniques 2026, Group XX.

This notebook covers Task 2 of the assignment. It loads the Expedia dataset, runs basic checks, builds the plots that go into the report, and ends with a short list of feature engineering suggestions for the modelling team.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')

DATA = Path('data')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 1. Load data and basic shape

In [ ]:
train = pd.read_csv(DATA / 'training_set_VU_DM.csv')
test = pd.read_csv(DATA / 'test_set_VU_DM.csv')

print('train shape:', train.shape)
print('test shape :', test.shape)
print('train cols not in test:', sorted(set(train.columns) - set(test.columns)))
train.head()

In [ ]:
train.info(memory_usage='deep')

In [ ]:
train.describe(include='all').T

## 2. Search-level overview

Each search (`srch_id`) has multiple rows, one per shown hotel. We want to know how many searches there are, how many hotels per search, and the click and book rates.

In [ ]:
n_searches_train = train['srch_id'].nunique()
n_searches_test = test['srch_id'].nunique()
n_hotels = train['prop_id'].nunique()

print(f'searches in train: {n_searches_train:,}')
print(f'searches in test : {n_searches_test:,}')
print(f'distinct hotels  : {n_hotels:,}')
print(f'click rate       : {train["click_bool"].mean():.4f}')
print(f'book rate        : {train["booking_bool"].mean():.4f}')
print(f'book given click : {train.loc[train.click_bool == 1, "booking_bool"].mean():.4f}')
print(f'random sort share: {train["random_bool"].mean():.4f}')

In [ ]:
search_sizes = train.groupby('srch_id').size()
print(search_sizes.describe())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(search_sizes.clip(upper=40), bins=range(0, 41), edgecolor='white')
ax.set_xlabel('hotels per search (clipped at 40)')
ax.set_ylabel('number of searches')
ax.set_title('Distribution of search sizes')
savefig('01_search_sizes')

## 3. Missing values

Several columns are mostly null (visitor history, competitor columns, query affinity, distance). The missingness pattern itself can be informative.

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing.to_string())

fig, ax = plt.subplots(figsize=(8, max(4, 0.25 * len(missing))))
missing.plot.barh(ax=ax, color='#4c72b0')
ax.invert_yaxis()
ax.set_xlabel('fraction missing')
ax.set_title('Missing values per column (train)')
savefig('02_missingness')

## 4. Position bias

The most important EDA finding for this dataset. Click rate drops sharply with position. The test set has no `position` column, which means we cannot use it as a feature directly. The `random_bool` flag lets us look at click rate by position when the sort is random, which removes the position effect.

In [ ]:
by_pos = train.groupby('position').agg(
    click_rate=('click_bool', 'mean'),
    book_rate=('booking_bool', 'mean'),
    n=('click_bool', 'size')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(by_pos['position'], by_pos['click_rate'], label='click rate', marker='o')
ax.plot(by_pos['position'], by_pos['book_rate'], label='book rate', marker='s')
ax.set_xlabel('position on results page')
ax.set_ylabel('rate')
ax.set_title('Click and book rate by position')
ax.legend()
savefig('03_position_bias')

In [ ]:
by_pos_rand = train.groupby(['random_bool', 'position'])['click_bool'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
for r, sub in by_pos_rand.groupby('random_bool'):
    label = 'random sort' if r == 1 else 'normal sort'
    ax.plot(sub['position'], sub['click_bool'], marker='o', label=label)
ax.set_xlabel('position')
ax.set_ylabel('click rate')
ax.set_title('Click rate by position, normal vs random sort')
ax.legend()
savefig('04_position_bias_random')

## 5. Price distribution and outliers

`price_usd` has extreme outliers. Different sites use different conventions (per night vs total stay, taxes included or not), which inflates this further.

In [ ]:
print(train['price_usd'].describe([0.5, 0.9, 0.95, 0.99, 0.999, 0.9999]))

low, high = train['price_usd'].quantile([0.01, 0.99])
trimmed = train['price_usd'].clip(low, high)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.log1p(train['price_usd'].clip(0, 1e7)), bins=80, edgecolor='white')
axes[0].set_title('log(1 + price_usd), full range')
axes[0].set_xlabel('log price')
axes[1].hist(trimmed, bins=80, edgecolor='white')
axes[1].set_title(f'price_usd, clipped to [{low:.0f}, {high:.0f}]')
axes[1].set_xlabel('price (USD)')
savefig('05_price_distribution')

In [ ]:
med = train.groupby('site_id')['price_usd'].median().sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
med.plot.bar(ax=ax, color='#4c72b0')
ax.set_xlabel('site_id')
ax.set_ylabel('median price_usd')
ax.set_title('Median displayed price by site_id')
savefig('06_price_by_site')

## 6. Star rating, review score and brand

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
train.groupby('prop_starrating')['click_bool'].mean().plot.bar(ax=axes[0], color='#4c72b0')
axes[0].set_title('Click rate by hotel star rating')
axes[0].set_xlabel('prop_starrating')
axes[0].set_ylabel('click rate')

train.groupby('prop_review_score')['click_bool'].mean().plot.bar(ax=axes[1], color='#dd8452')
axes[1].set_title('Click rate by review score')
axes[1].set_xlabel('prop_review_score')
savefig('07_star_review_clickrate')

In [ ]:
tab = train.groupby(['prop_brand_bool', 'promotion_flag'])[['click_bool', 'booking_bool']].mean()
print(tab)

## 7. Within-search relative features

Users compare hotels inside one search, not across all of Expedia. Relative position of price within a search is a stronger predictor than absolute price.

In [ ]:
tmp = train[['srch_id', 'price_usd', 'click_bool', 'booking_bool']].copy()
tmp['price_q'] = tmp.groupby('srch_id')['price_usd'].transform(
    lambda s: pd.qcut(s.rank(method='first'), q=min(10, max(1, s.nunique())), labels=False, duplicates='drop')
)
rel = tmp.groupby('price_q')[['click_bool', 'booking_bool']].mean()
print(rel)

fig, ax = plt.subplots(figsize=(7, 4))
rel.plot.bar(ax=ax)
ax.set_xlabel('price decile within search (0 = cheapest)')
ax.set_ylabel('rate')
ax.set_title('Click and book rate by within-search price decile')
savefig('08_within_search_price')

## 8. Search context features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col, label in zip(
    axes.ravel(),
    ['srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count'],
    ['length of stay (nights)', 'booking window (days)', 'adults', 'children'],
):
    upper = train[col].quantile(0.99)
    ax.hist(train[col].clip(upper=upper), bins=30, edgecolor='white')
    ax.set_title(label)
savefig('09_search_context')

## 9. Correlations among numeric features

In [ ]:
num_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price',
    'price_usd', 'promotion_flag',
    'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count',
    'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool',
    'click_bool', 'booking_bool',
]
num_cols = [c for c in num_cols if c in train.columns]
corr = train[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 7}, ax=ax)
ax.set_title('Correlation among numeric features')
savefig('10_correlation')

## 10. Train / validation split for the modelling team

Splitting by row breaks the search structure. Always split by `srch_id` so a search is fully in train or fully in validation.

In [ ]:
def split_by_search(df, val_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    sids = df['srch_id'].unique()
    rng.shuffle(sids)
    cut = int(len(sids) * (1 - val_frac))
    train_ids = set(sids[:cut])
    train_part = df[df['srch_id'].isin(train_ids)].copy()
    val_part = df[~df['srch_id'].isin(train_ids)].copy()
    return train_part, val_part

tr, va = split_by_search(train, val_frac=0.2)
print(f'train searches: {tr["srch_id"].nunique():,}, rows: {len(tr):,}')
print(f'valid searches: {va["srch_id"].nunique():,}, rows: {len(va):,}')

## 11. Findings and feature engineering suggestions for the modelling team

_(Filled in by the EDA author after running the cells above.)_

**Headline findings**
1. Position bias is severe. Click rate at position 1 is roughly N times that at position 5.
2. Visitor history and competitor columns are mostly null. Treat the null itself as a feature.
3. Price has extreme outliers and varies by `site_id`, suggesting currency or display convention differences.
4. Within-search price rank predicts clicks better than absolute price.
5. Star rating and review score both correlate positively with click rate, with diminishing returns above 4 stars.

**Suggested engineered features**
- `missing_visitor_hist` flag and median imputation for `visitor_hist_starrating` and `visitor_hist_adr_usd`.
- Per-property aggregates from train: mean click rate, mean book rate, mean position, mean price.
- Within-search normalisations: price rank, star rating rank, log price minus search mean log price.
- Price difference from `prop_log_historical_price`.
- Counts of available competitor data, mean of `comp*_rate` and `comp*_inv` ignoring nulls.
- Date features from `date_time`: month, weekday, season.
- Drop or cap `price_usd` above the 99th percentile.

**Important constraints**
- `position` cannot be used as a feature for prediction since it is absent in the test set.
- Train/validation split must be by `srch_id`, not by row.
- The Kaggle metric is NDCG@5 with relevance grades 5 (book), 1 (click), 0 (neither). Loss should reflect this, for example LightGBM `lambdarank` with custom labels.